# 03 NH투자증권(2018) 퀄리티·로우볼 팩터 재현

## 0. 원문 정리
원문: NH투자증권 하재석·박소현(2018-10-18), 「Korea Factor Investing 시리즈(2): 버핏의 알파, 퀄리티와 로우볼」(`마법공식_자료\2_증권사리포트_근거\01_NH_2018-10-18_KoreaFactorInvesting2_퀄리티와로우볼.pdf`). 퀄리티 시험 조건(p.18), 로우볼 시험 조건(p.23), 시총 300위(p.20·25), 리밸런싱 주기(p.21·26), 변동성 산출 기간·주기(p.23).

원칙(회원 결정 2026-11-06): **원문에 있으면 원문대로, 원문에 말이 없으면 뺀다.** 원문에 없던 우리 추가(성과표·확인판·검정 등)는 `연구\따라하기에서_옮김_03.ipynb`로 옮겼다. 원문에 없는데 빼면 계산이 안 되는 칸은 기본 방법(① 자료가 없거나 계산이 안 되면 그 종목을 그 달에서 뺀다 ② 나머지는 가장 단순한 쪽)으로 정하고 10절에 목록으로 남겼다.

| 항목 | 원문(쪽·인용) | 우리 설정 |
|---|---|---|
| 유니버스 | p.18·23 'KOSPI 상장종목가운데관리종목, 거래정지상태가아닌종목. 생존편향을제거하기위해리밸런싱당일상장되어있던종목을대상으로함' | `시장=["KS"]`, 거래정지(CP10001650)·관리종목(CP10001660) 제외, 리밸런싱일 상장 종목(상장폐지 포함 자료) |
| 보통주 | 원문에 말 없음 | **보통주만**(코드 끝자리 0, 회원 결정 2026-11-06 "보통주만 해") |
| 수익률 가격 | 원문에 말 없음 | **수정주가 S410000700**(배당 미포함, 회원 결정 2026-11-06) |
| 퀄리티 지표 | p.18 'ROE, 영업이익률, 부채비율, 총자산회전율' | `ROE`·`영업이익률`·`부채비율`(낮을수록 1분위)·`총자산회전율`. 정의(분자·분모)는 원문에 없어 10절 |
| 로우볼 지표 | p.23 '일간수익률의12개월변동성하위' | `변동성` = 직전 12개월 일간수익률 표준편차, 낮을수록 1분위 |
| 묶음 | p.18 '지표기준1분위에서5분위포트폴리오의수익률을동일가중으로측정' | `고르기=("분위",5)`, `비중=["동일"]`(로우볼 상자 p.23에는 가중 말이 없어 같은 동일가중, 10절) |
| 리밸런싱 | p.18·23 '리밸런싱: 3개월단위월말리밸런싱' | `기준일="마지막거래일"`, `리밸런싱=3`. 몇 월인지는 원문에 없어 기간 첫 달(2002-03)부터 3개월마다 = 3·6·9·12월 말(10절) |
| 기간 | p.18·23 '시뮬레이션기간2002년3월~ 2018년9월' | `기간=("2002-03","2018-09")`: 2002-03 월말 첫 리밸런싱, 2018-09 월말까지 |
| 재무 반영 | p.18 '주요재무제표수치는분기/연간재무제표가발표되는월의월말에반영' | `재무주기=("발표값",)`(`포.발표재무`): 그날까지 발표된 가장 최근 분기/연간 보고서 값 **그대로**(4분기 합·×4 없음, 회원 결정 2026-11-06). 발표일(FP56000900) 당일부터 쓸 수 있어, 월말 리밸런싱이면 그 달 마지막 거래일까지 발표분이 그 달 말에 들어감. 3개월 리밸런싱이라 5·8·11월에 나온 분기 보고서는 다음 리밸런싱(6·9·12월 말)에 들어감 |
| 결과 | p.18 '4가지지표모두2~ 4분위포트폴리오의누적수익률이가장높았다', '로우퀄리티 바스켓의성과는공통적으로현저히낮은성과'; p.18 ROE L/S '2008년금융위기에도양호한성과…시장과낮은상관관계'; p.23 로우볼 프리미엄·2008년 헤지; p.20·25 시총 300위; p.21·26 리밸런싱 주기; p.23·26 변동성 산출 기간·주기 | 9절에서 원문 문장마다 방향을 판정(판정 기준은 9절 머리에 먼저 적음) |

그림: 원문 그림처럼 누적 수익률(%)을 보통 축으로 그린다(그림 22~27·29·32~35·37). 원문 그림의 선 값은 읽을 수 없어 숫자 대조는 하지 않는다.

줄임말: L/S·LS(롱숏: 1분위를 사고 5분위를 판 수익률, 매달 Q1 − Q5를 복리로 쌓음), Q1~Q5(1~5분위).

## 1. 불러오기
계산은 모두 `공통/` 폴더의 모듈이 한다. 이 노트북은 설정을 정하고, 실행하고, 결과를 본다.
짧은 시험: 환경변수 `FOLLOW_TEST=1`이면 기간을 2002-03~2004-02(약 2년)로 줄이고 결과 파일을 `_검증\1106\시험결과\`에만 쓴다(실제 결과 파일을 덮어쓰지 않음).

In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "공통"))
import numpy as np
import pandas as pd
import 데이터읽기 as 읽기, 종목거르기, 지표, 포트폴리오 as 포, 평가
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
시험 = os.environ.get("FOLLOW_TEST") == "1"
if 시험:  # 저장 경로를 시험 폴더로(평가.저장·판정 csv 모두 평가.결과폴더를 씀)
    평가.결과폴더 = Path.cwd().parent / "_검증" / "1106" / "시험결과"
    평가.결과폴더.mkdir(parents=True, exist_ok=True)
print("시험 모드:", 시험, "/ 결과 폴더:", 평가.결과폴더)

## 2. 데이터 경로와 기간
기간은 (첫 리밸런싱 월, 마지막 기준일 월)이다. 원문 p.18·23 '시뮬레이션기간2002년3월~ 2018년9월'.

In [ ]:
기간 = ("2002-03", "2004-02") if 시험 else ("2002-03", "2018-09")
print("데이터:", 읽기.DATA, "/ 기간:", 기간)

## 3. 설정
원문 p.18·23 시험 조건 그대로. 원문에 없는 칸(10절)은 기본 방법. 재무시점(`재무시점`)은 `재무주기=("발표값",)`에서 쓰이지 않는다(발표일 당일부터).

In [ ]:
기본 = {**포.기본설정, "이름": "NH퀄리티로우볼", "시장": ["KS"], "기간": 기간, "기준일": "마지막거래일", "리밸런싱": 3,
       "가격": 읽기.수정주가, "재무주기": ("발표값",),
       "제외": ["보통주", ("거래정지", {"아이템": 읽기.정지여부}), "관리종목"],
       "고르기": ("분위", 5), "비중": ["동일"], "거래비용": dict(수수료=0, 거래세=0, 슬리피지=0)}
지표들 = {"ROE": ("ROE", "높게"), "영업이익률": ("영업이익률", "높게"), "부채비율": ("부채비율", "낮게"),
         "총자산회전율": ("총자산회전율", "높게"), "로우볼": ("변동성", "낮게")}
퀄리티 = ["ROE", "영업이익률", "부채비율", "총자산회전율"]
분위 = [f"Q{k}" for k in range(1, 6)]
코스피 = 읽기.지수()["I.001"].reindex(포.기준일(기본))  # 리밸런싱 기준일(월말) KOSPI 종가지수: 원문 그림 26·32 'KOSPI(우)'
코스피월 = 코스피.pct_change(fill_method=None)
누적값 = lambda r: (1 + r.dropna()).prod() - 1 if r.notna().any() else np.nan  # 값이 없는 구간은 빈칸(0으로 안 보임)
해누적 = lambda r, y: (1 + r[r.index.year == y].dropna()).prod() - 1 if (r.index.year == y).any() else np.nan  # 그해 1~12월 월수익률 누적(그해가 없으면 빈칸)
pd.Series(기본).astype(str).to_frame("공통 설정")

## 4. 실행: 지표별 1~5분위 (그림 22~25)
원문 p.19 그림 22~25: 지표별 1~5분위 누적 수익률(%).

In [ ]:
결과 = {n: 포.실행({**기본, "지표": [m]}) for n, m in 지표들.items()}
수익 = {n: o["수익률"]["동일"] for n, o in 결과.items()}
누적 = pd.DataFrame({n: (1 + r).prod() - 1 for n, r in 수익.items()}).T
for i, n in enumerate(퀄리티):
    평가.누적그림(수익[n][분위].rename(columns=lambda g: f"{g[1]}분위"), f"그림 {22 + i}. {n} 지표 기준 퀄리티 포트폴리오 성과 추이", 퍼센트=True)
누적

## 5. ROE·로우볼 L/S와 KOSPI (그림 26·32), 2008년
- 원문 p.19 그림 26 'ROE 기준퀄리티Long/Short 포트폴리오는시장하락시기방어적인효과를보임'(L/S 좌축 %, KOSPI 우축 P), p.18 '2008년금융위기에도양호한성과를기록하는등, 시장과낮은상관관계를보였다'.
- 원문 p.24 그림 32 'KOSPI 로우볼Long/Short 포트폴리오성과추이'(L/S 좌축 %, KOSPI 우축 P), p.23 '2008년금융위기에로우볼Long/Short 포트폴리오는시장에대한헤지역할을수행'.
- 2008년 = 2008년 1~12월 월수익률 누적. 상관 = L/S 월수익률과 KOSPI 월수익률의 상관(전 기간). 원문은 상관 숫자를 주지 않아 숫자만 보인다.

In [ ]:
LS = pd.DataFrame({n: 수익[n]["Q1"] - 수익[n]["Q5"] for n in ["ROE", "로우볼"]})
평가.누적그림(LS[["ROE"]].set_axis(["1분위-5분위 Long/Short(좌)"], axis=1), "그림 26. ROE 기준 퀄리티 Long/Short와 KOSPI", 퍼센트=True, 오른쪽={"KOSPI": 코스피})
평가.누적그림(LS[["로우볼"]].set_axis(["1분위-5분위 Long/Short(좌)"], axis=1), "그림 32. KOSPI 로우볼 Long/Short 포트폴리오 성과 추이", 퍼센트=True, 오른쪽={"KOSPI": 코스피})
표2008 = pd.DataFrame({"2008년 누적": {"ROE L/S": 해누적(LS["ROE"], 2008), "로우볼 L/S": 해누적(LS["로우볼"], 2008), "KOSPI": 해누적(코스피월, 2008)},
                       "KOSPI 월수익률과 상관(전 기간)": LS.corrwith(코스피월).rename({"ROE": "ROE L/S", "로우볼": "로우볼 L/S"})})
표2008

## 6. 시가총액 300위 이내 (그림 27·35)
- 원문 p.20 'KOSPI 상장전체종목과시가총액상위300위이내종목의ROE 기준1분위와5분위포트폴리오성과를비교한결과유니버스에따른성과차이가미미했다'(그림 27).
- 원문 p.25 '2002년이후시가총액300위이내종목의로우볼Long/Short 포트폴리오와KOSPI 전체 Long/Short 포트폴리오의누적성과는유사했으나 2007~ 20014년까지 KOSPI 전체유니버스의로우볼팩터가다소우수'(그림 35. '20014년'은 원문 오타로 보고 2014년으로 읽음).
- 300위 = 리밸런싱일 시가총액(S410001250) 상위 300종목, 다른 거르기(보통주·거래정지·관리종목) 뒤(10절).
- L/S 누적 계산법은 원문에 글이 없어 매달 Q1 − Q5 차이를 복리로 쌓는다(5·7·8절도 같음, 10절). 원문 그림 27의 'KOSPI 전체' 끝값(눈대중 약 230%)은 그림 22의 1분위 누적 − 5분위 누적(약 510 − 280)과 맞아 보이고, 같은 ROE L/S인 그림 26(끝 약 0%)과는 원문 안에서 서로 맞지 않는다(`_검증\1106\원문사양_03.md` E절, 눈대중).

In [ ]:
대형 = {n: 포.실행({**기본, "지표": [지표들[n]], "제외": 기본["제외"] + [("시총상위", {"n": 300})]})["수익률"]["동일"] for n in ["ROE", "로우볼"]}
대형LS = pd.DataFrame({n: r["Q1"] - r["Q5"] for n, r in 대형.items()})
평가.누적그림(pd.DataFrame({"1분위-5분위 L/S(KOSPI 전체종목)": LS["ROE"], "1분위-5분위 L/S(시가총액 300위 이내)": 대형LS["ROE"]}), "그림 27. ROE 기준 L/S: KOSPI 전체 대 시가총액 300위 이내", 퍼센트=True)
평가.누적그림(pd.DataFrame({"시가총액 300위 이내 L/S": 대형LS["로우볼"], "KOSPI 전체 L/S": LS["로우볼"]}), "그림 35. 로우볼 L/S: KOSPI 전체 대 시가총액 300위 이내", 퍼센트=True)
대형표 = pd.DataFrame({f"{n} L/S": {"KOSPI 전체 누적": 누적값(LS[n]), "300위 이내 누적": 누적값(대형LS[n]),
                                    "KOSPI 전체 2007~2014 누적": 누적값(LS[n].loc["2007":"2014"]), "300위 이내 2007~2014 누적": 누적값(대형LS[n].loc["2007":"2014"])} for n in ["ROE", "로우볼"]})
대형표

## 7. 리밸런싱 주기 (그림 29·37)
- 원문 p.21 'KOSPI 대상으로3개월, 6개월단위리밸런싱에따른퀄리티팩터의성과차이를분석한결과차이가거의나타나지않았다'(그림 29: 총자산회전율 1분위·5분위, 3·6개월).
- 원문 p.26 '3개월, 6개월, 12개월단위리밸런싱을하는로우볼Long/Short 포트폴리오성과를비교한결과3개월단위리밸런싱에비해6개월, 12개월단위리밸런싱의누적성과가우수했다. 다만, 6개월과12개월단위리밸런싱포트폴리오는유사한성과'(그림 37).
- 6·12개월도 같은 기간 첫 달(2002-03)부터 센다(10절).

In [ ]:
회전 = {k: 수익["총자산회전율"] if k == 3 else 포.실행({**기본, "지표": [지표들["총자산회전율"]], "리밸런싱": k})["수익률"]["동일"] for k in (3, 6)}
볼주기 = {k: 수익["로우볼"] if k == 3 else 포.실행({**기본, "지표": [지표들["로우볼"]], "리밸런싱": k})["수익률"]["동일"] for k in (3, 6, 12)}
평가.누적그림(pd.DataFrame({"1분위(3개월 리밸런싱)": 회전[3]["Q1"], "1분위(6개월 리밸런싱)": 회전[6]["Q1"], "5분위(6개월 리밸런싱)": 회전[6]["Q5"], "5분위(3개월 리밸런싱)": 회전[3]["Q5"]}),
            "그림 29. 리밸런싱 주기에 따른 퀄리티 포트폴리오(총자산회전율)", 퍼센트=True)
볼주기LS = pd.DataFrame({f"{k}개월 리밸런싱": r["Q1"] - r["Q5"] for k, r in 볼주기.items()})
평가.누적그림(볼주기LS, "그림 37. 리밸런싱 주기에 따른 로우볼 Long/Short 누적 성과", 퍼센트=True)
display(pd.DataFrame({f"{k}개월": {"1분위 누적": 누적값(r["Q1"]), "5분위 누적": 누적값(r["Q5"])} for k, r in 회전.items()}).rename_axis("총자산회전율"))
(1 + 볼주기LS).prod().sub(1).rename("로우볼 L/S 누적").to_frame()

## 8. 변동성 산출 기간·주기 (그림 33·34)
- 원문 p.23 '최근12개월주가변동성대신최근24개월, 36개월일간수익률의변동성을기준으로로우볼팩터성과를비교한결과주가변동성산출기간에따른영향은거의없었다'(그림 33).
  **원문 안에서 어긋남**: 본문은 '24개월, 36개월'이고 그림 33 범례는 '12개월변동성·36개월변동성·60개월변동성'이다. 본문 문장이 기준(12·24·36개월)이고, 그림 범례도 원문이라 60개월을 더해 12·24·36·60개월을 모두 돌린다(`_검증\1106\원문사양_03.md` C절).
- 원문 p.23 '일간수익률대신주간수익률, 월간수익률기준으로백테스트한결과도유사한결과'(그림 34: Daily·Weekly·Monthly 주가수익률 변동성). 주간·월간은 직전 12개월.
- `지표.변동성_창(s, 개월, 주기)`(공통): 창 내내 시장구분이 코스피·코스닥이고 창 첫날 상장 가격이 있는 종목만, 주간 = 금요일 끝 가격, 월간 = 월말 가격(10절). 12개월·일간이면 `지표.변동성`과 같은 값.

In [ ]:
for k in (24, 36, 60):  # 실행이 지표 이름(문자열)으로 함수를 찾으므로 이 노트북 안에서만 이름을 붙인다(공통 파일은 그대로)
    setattr(지표, f"변동성_{k}개월", lambda s, k=k: 지표.변동성_창(s, k, "일"))
for p in ("주", "월"):
    setattr(지표, f"변동성_{p}간", lambda s, p=p: 지표.변동성_창(s, 12, p))
볼실행 = lambda m: 포.실행({**기본, "지표": [(m, "낮게")]})["수익률"]["동일"]
창LS = pd.DataFrame({"12개월(본문)": LS["로우볼"], **{f"{k}개월": (lambda r: r["Q1"] - r["Q5"])(볼실행(f"변동성_{k}개월")) for k in (24, 36, 60)}})
주기LS = pd.DataFrame({"Daily": LS["로우볼"], **{n: (lambda r: r["Q1"] - r["Q5"])(볼실행(f"변동성_{p}간")) for n, p in (("Weekly", "주"), ("Monthly", "월"))}})
평가.누적그림(창LS, "그림 33. 변동성 산출 기간에 따른 로우볼 Long/Short(본문 12·24·36개월, 범례 60개월)", 퍼센트=True)
평가.누적그림(주기LS, "그림 34. 일간·주간·월간 변동성 기준 로우볼 Long/Short", 퍼센트=True)
pd.concat({"산출 기간": (1 + 창LS).prod() - 1, "수익률 주기": (1 + 주기LS).prod() - 1}).rename("로우볼 L/S 누적").to_frame()

## 9. 원문 vs 우리
원문은 숫자 표 없이 그림과 문장이라, 문장이 우리 결과에서도 같은 방향인지 참/거짓으로 본다. 판정 기준(돌리기 전에 정함, 원문이 말한 방향·크기만):

| 번호 | 원문(쪽·인용) | 판정 기준 |
|---|---|---|
| 1 | p.18 '4가지지표모두2~ 4분위포트폴리오의누적수익률이가장높았다' | 4개 퀄리티 지표 모두 누적 최고 분위가 Q2~Q4 |
| 2 | p.18 '로우퀄리티 바스켓의성과는공통적으로현저히낮은성과' | 4개 지표 모두 Q5 누적이 가장 낮음('현저히'는 숫자 기준이 없어 넣지 않음) |
| 3 | p.18 ROE L/S '2008년금융위기에도양호한성과', 그림 26 '시장하락시기방어적인효과' | KOSPI 2008년 < 0 이고 ROE L/S 2008년 누적 > 0 |
| 4 | p.23 'KOSPI에도로우볼프리미엄이존재' | 로우볼 L/S 누적 > 0 |
| 5 | p.23 '2008년금융위기에로우볼Long/Short 포트폴리오는시장에대한헤지역할' | KOSPI 2008년 < 0 이고 로우볼 L/S 2008년 누적 > 0 |
| 6 | p.25 '2007~ 20014년까지KOSPI 전체유니버스의로우볼팩터가다소우수' | 2007~2014 로우볼 L/S 누적: KOSPI 전체 > 300위 이내 |
| 7 | p.25 '로우볼팩터역시중, 소형주와대형주를가리지않고상당한팩터프리미엄', p.26 요약 2) | 300위 이내 로우볼 L/S 누적 > 0 |
| 8 | p.26 '3개월단위리밸런싱에비해6개월, 12개월단위리밸런싱의누적성과가우수' | 로우볼 L/S 누적: 6개월·12개월 모두 > 3개월 |
| 9 | p.26 요약 1) '변동성산출기간이나수익률주기(일간, 주간, 월간) 등에관계없이로우볼프리미엄이존재' | 12·24·36·60개월, 일간·주간·월간 로우볼 L/S 누적이 모두 > 0 |

숫자 기준이 없어 판정하지 않고 숫자만 보이는 문장: '시장과낮은상관관계'(p.18, 5절 상관), 시총 300위 ROE '성과차이가미미'(p.20, 6절), 퀄리티 3·6개월 '차이가거의나타나지않았다'(p.21, 7절), 변동성 산출 기간·주기 '영향은거의없었다'(p.23, 8절), '6개월과12개월…유사한성과'(p.26, 7절), 로우볼 300위·전체 '누적성과는유사'(p.25, 6절).

In [ ]:
최고 = {n: 누적.loc[n, 분위].idxmax() for n in 퀄리티}
최저 = {n: 누적.loc[n, 분위].idxmin() for n in 퀄리티}
볼누적 = (1 + 볼주기LS).prod() - 1
변형누적 = pd.concat([(1 + 창LS).prod() - 1, (1 + 주기LS).prod() - 1])
판정표 = pd.DataFrame({
    "1. 4개 퀄리티 지표 모두 2~4분위 누적이 가장 높음": ("p.18", all(g in ["Q2", "Q3", "Q4"] for g in 최고.values()), "누적 최고 " + ", ".join(f"{n} {g}" for n, g in 최고.items())),
    "2. 4개 퀄리티 지표 모두 5분위 누적이 가장 낮음": ("p.18", all(g == "Q5" for g in 최저.values()), "누적 최저 " + ", ".join(f"{n} {g}" for n, g in 최저.items())),
    "3. 2008년 KOSPI < 0 이고 ROE L/S > 0(양호)": ("p.18·19", bool(표2008.loc["KOSPI", "2008년 누적"] < 0 and 표2008.loc["ROE L/S", "2008년 누적"] > 0),
                                           f"ROE L/S {표2008.loc['ROE L/S', '2008년 누적']:.4f}, KOSPI {표2008.loc['KOSPI', '2008년 누적']:.4f}"),
    "4. 로우볼 L/S 누적 > 0": ("p.23", bool(누적값(LS["로우볼"]) > 0), f"{누적값(LS['로우볼']):.4f}"),
    "5. 2008년 KOSPI < 0 이고 로우볼 L/S > 0(헤지)": ("p.23·24", bool(표2008.loc["KOSPI", "2008년 누적"] < 0 and 표2008.loc["로우볼 L/S", "2008년 누적"] > 0),
                                                 f"로우볼 L/S {표2008.loc['로우볼 L/S', '2008년 누적']:.4f}, KOSPI {표2008.loc['KOSPI', '2008년 누적']:.4f}"),
    "6. 2007~2014 로우볼 L/S 누적: 전체 > 300위 이내": ("p.25", bool(대형표.loc["KOSPI 전체 2007~2014 누적", "로우볼 L/S"] > 대형표.loc["300위 이내 2007~2014 누적", "로우볼 L/S"]),
                                                    f"전체 {대형표.loc['KOSPI 전체 2007~2014 누적', '로우볼 L/S']:.4f}, 300위 {대형표.loc['300위 이내 2007~2014 누적', '로우볼 L/S']:.4f}"),
    "7. 300위 이내 로우볼 L/S 누적 > 0": ("p.25·26", bool(대형표.loc["300위 이내 누적", "로우볼 L/S"] > 0), f"{대형표.loc['300위 이내 누적', '로우볼 L/S']:.4f}"),
    "8. 로우볼 L/S 누적: 6·12개월 리밸런싱 > 3개월": ("p.26", bool(min(볼누적["6개월 리밸런싱"], 볼누적["12개월 리밸런싱"]) > 볼누적["3개월 리밸런싱"]),
                                                  ", ".join(f"{k} {v:.4f}" for k, v in 볼누적.items())),
    "9. 변동성 12·24·36·60개월, 일·주·월간 모두 로우볼 L/S 누적 > 0": ("p.23·26", bool((변형누적 > 0).all()), ", ".join(f"{k} {v:.4f}" for k, v in 변형누적.items())),
}, index=["원문 쪽", "우리", "우리 숫자"]).T.rename_axis("원문 주장")
판정표.to_csv(평가.결과폴더 / "03_원문판정.csv", encoding="utf-8-sig")  # 결과요약 노트북이 읽음
판정표

## 10. 원문이 말하지 않아 기본 방법으로 정한 것
원문에 말이 없는데 빼면 계산이 안 되는 칸. 기본 방법 ① 자료가 없거나 계산이 안 되면 그 종목을 그 달에서 뺀다(우동호 p.512 '필요한 재무자료를 입수 가능한 기업' 원리) ② 나머지는 가장 단순한 쪽(회원 결정 2026-11-06).

| 항목 | 원문에 없음(확인한 곳) | 쓴 기본 방법 | 칸 |
|---|---|---|---|
| 리밸런싱 달 | p.18·23은 '3개월단위월말리밸런싱'과 '2002년3월~'뿐, 몇 월인지 없음(PDF 전체 검색) | ② 기간 첫 달 2002-03 월말부터 3개월마다 = 3·6·9·12월 말 | 3절 `리밸런싱=3`, 2절 `기간` |
| 기간 시작 해석 | '2002년3월'이 첫 리밸런싱 달인지 첫 수익률 달인지 없음(그림 가로축 '02.3부터) | ② 2002-03 월말 첫 리밸런싱(첫 수익률 달 2002-04) | 2절 |
| 6·12개월 리밸런싱 시작 달 | p.21·26 주기만 | ② 3개월과 같은 2002-03 월말부터 | 7절 |
| 로우볼 가중 | p.23 시험 조건 상자에 가중 없음 | ② 동일가중(퀄리티 p.18과 같음) | 3절 `비중` |
| 변동성 최소 관측 | p.23 '12개월변동성'뿐 | ① 12개월 변동성을 낼 수 없는 종목(직전 12개월 내내 시장구분이 코스피·코스닥이 아니거나 창 첫날 상장 가격이 없음)은 뺌(`지표.변동성`), 창 안 빈칸은 빼고 표준편차. 원문사양 C절은 '최소 관측 기준 없음(창 안 값으로 계산)'을 기본 ②로 적음 — 공통 함수 규칙이라 그대로 두고 차이로 적음 | 공통 `지표.변동성`·`변동성_창` |
| 1년 상장 조건을 퀄리티에도 걸지 | 원문 없음 | ② 로우볼(변동성)에만. 퀄리티 지표는 변동성을 안 써서 조건 없음 | 공통 |
| 거래정지 날 수익률 | 원문 없음 | ② 가격 그대로(정지 날 가격이 그대로면 수익률 0이 들어감) | 공통 `포.재료` |
| ROE 정의 | p.18 이름만 | ② 지배주주순이익(M000908004) / 지배주주자본(M000903003), 같은 보고서 값 | 공통 `지표.ROE` |
| 영업이익률 정의 | p.18 이름만 | ② 영업이익(M000906001) / 매출액(M000904001) | 공통 `지표.영업이익률` |
| 부채비율 정의 | p.18 이름만(방향은 '로우퀄리티' = 5분위) | ② 부채총계(M000902001) / 자본총계(M000903001), 낮을수록 1분위 | 공통 `지표.부채비율` |
| 총자산회전율 정의 | p.18 이름만 | ② 매출액 / 자산총계(보고서 기간말), 같은 보고서 값 | 공통 `지표.총자산회전율` |
| 음수 자본 | 원문 없음 | ② 공식 그대로(자본이 음수면 ROE·부채비율 부호가 뒤집힘). 분모 0(±무한대)만 ①로 뺌 | 공통 `포.실행` |
| 발표일 없는 보고서 | 원문 없음 | ① 안 씀(그 보고서를 건너뛰고 그 전에 발표된 보고서를 씀, 없으면 그 종목을 뺌) | 공통 `포.발표행` |
| 묵은 재무 상한 | 원문 없음 | ② 상한 없음(가장 최근 발표 보고서) | 공통 `포.발표재무(상한=None)` |
| 같은 날 발표된 보고서 둘 | 원문 없음 | ② 기간 끝이 늦은 보고서 | 공통 `포.발표재무` |
| 마지막 거래일 뒤 휴일 발표 | p.18 '발표되는월의월말'만 | 원문 문장대로 그 달(달력 달) 발표분이면 그 달 말에 씀(V03 HIGH-1 2026-11-06 고침; 리밸런싱일 종가에는 몰랐던 값 = 며칠 미래 정보, 한계. 2007-03-30에 연간 보고서 66 → 2,016종목, KS+KQ) | 공통 `포.발표재무` |
| 리밸런싱 사이 비중 | 원문 없음 | ② 사서 그대로 둠(매수보유) | 공통 `포.보유` |
| 보유 중 상장폐지 | p.18 생존편향 문장뿐 | ② 마지막 상장 가격으로 판 것으로 봄 | 공통 `포.기간수익률` |
| 분위 나누기·같은 값 | p.18 '1분위에서5분위'만 | ② 순위('first')로 종목 수를 같게 5등분 | 공통 `포.고르기` |
| 거래비용 | 시험 조건 상자(p.18·23)에 없음 | ② 0 | 3절 `거래비용` |
| L/S 누적 | p.18 'Long…Short'만(계산법 글 없음) | ② 매달 Q1 − Q5 수익률을 복리로 쌓음. 원문사양 E절: 그림 27 끝값은 'Q1 누적 − Q5 누적'과 맞아 보임(눈대중), 그림 26은 그림 27과 서로 안 맞음 | 5·6·7·8절 |
| 시총 300위 순서·날짜 | p.20·25 '시가총액상위300위이내'만 | ② 리밸런싱일 시가총액, 다른 거르기 뒤 상위 300 | 6절 |
| 변동성 24·36·60개월, 주간·월간 | p.23 기간·주기만 | ② `지표.변동성`과 같은 '창 내내 상장' 규칙, 주간 = 금요일 끝, 월간 = 월말 가격 | 8절 |
| 2008년·상관 계산 | p.18·23 '2008년', '낮은상관관계'만 | ② 2008년 1~12월 월수익률 누적, 월수익률 상관(전 기간) | 5절 |
| KOSPI(우) | 그림 26·32 'KOSPI(우)'(P) | ② 리밸런싱 기준일(월말) KOSPI 종가지수 I.001 | 3·5절 |

## 11. 차이 이유 후보
- **재무 금액**: 발표된 보고서 값 그대로라 분기 보고서(1·2·3분기) 행의 순이익·영업이익·매출은 3개월 값, 사업보고서 행은 1년 값이다. 같은 리밸런싱일에 종목마다 최근 보고서 종류가 다르면(예: 3월 말에 사업보고서를 아직 안 낸 종목) ROE·총자산회전율의 크기가 서로 다른 기간 값으로 비교된다. 원문이 어떻게 했는지는 원문에 없다.
- **리밸런싱 달**: 3·6·9·12월 말이라 5·8·11월에 나온 분기 보고서는 한 달 뒤 리밸런싱에 들어간다(10절).
- **ROE·부채비율 정의**: 원문에 분모(기말/평균, 지배/전체) 정의가 없다(10절).
- **음수 분모**: ROE·부채비율은 공식 그대로라 자본이 음수인 종목은 부호가 뒤집혀 좋은 쪽에 줄 설 수 있다(값은 고치지 않음).
- **변동성**: 원문은 12개월 일간 변동성. 원문에 최소 관측 기준이 없어 직전 12개월 내내 상장한 종목만 쓴다.
- **자료 판본**: 원문(2018)은 그때 FnGuide 자료, 우리는 DataGuide 2026-11 판(재무는 최근 수정치).
- **벤치마크**: 그림의 KOSPI는 가격지수(I.001, 배당 미포함), 포트폴리오 수익률도 수정주가(배당 미포함)라 같은 기준이다.

## 12. 자료 처리(원문에 없는 자료 사정)
| 항목 | 우리 | 이유 |
|---|---|---|
| 상장 구간 | 시장구분(CP10000300)이 코스피·코스닥('KS'·'KQ')인 날의 가격만 계산에 씀(`읽기.상장가격`). 원본 값은 그대로 | 통합_확인기록 F2(상장일보다 앞선 가격 = 이전상장 전 시장 가격)·F3(상장폐지 뒤 장외 가격). 회원 결정 2026-11-02 |
| '쓰지 말 것' 칸 | 리밸런싱일이 README_데이터의 '쓰지 말 것' 구간 (시작, 끝]에 드는 종목은 편입 대상에서 뺌(`종목거르기.쓰지말것`). 보유 중 걸린 칸은 빈칸(`포.쓰지말것칸`). 직전 12개월 일간수익률에서도 빈칸으로 봄. 값은 그대로. 수정주가를 쓰므로 해당 칸은 A008080 2004-01~2013-09(D-05)와 수정주가 병합·감자 미반영 7칸의 그 달(D27) | 회원 결정 2026-11-02 |
| 끝 날짜 | 가격·시장 일간 항목(S·CP 파일)에 값이 있는 마지막 날(`읽기.공통끝`)에서 자름 | 통합_확인기록 F6 |
| 코드 재사용 16종목 | 재사용 경계를 넘는 수익률을 쓰지 않음(공백 구간이 빈칸이라 끊김). 직전 12개월 상장 판정도 시장구분으로 함 | 통합_확인기록 F5, README_데이터 D-04 |
| 재무 값 뽑기 | 분기 보고서 행 = 달력기준 일간 분기(3개월) 재무의 분기말 값(`읽기.분기재무`), 사업보고서 행 = 일간 결산 재무의 결산일 값(`읽기.연간재무`). 쓸 수 있는 날 = 그 보고서 발표일(FP56000900) | 일간 값은 발표 전부터 그 기간 값이 들어 있어 그대로 쓰면 미래 정보 |

## 13. 요약표용 저장
지표별 Q1과 원문에 L/S가 있는 ROE·로우볼 L/S의 월수익률을 `결과_월수익률/`에 남긴다(`연구\요약표.ipynb`가 읽음). 요인(`평가.요인`)과 이상값(+300% 종목-월 보유 건수, L/S는 Q1 + Q5)은 요약표가 쓰는 저장용이고 이 노트북의 원문 대조에는 쓰지 않는다.

In [ ]:
F = 평가.요인(포.기준일(기본), 기본)
이상 = {n: 포.이상값보유(o) for n, o in 결과.items()}
S = pd.DataFrame({(f"NH퀄리티로우볼 {n} Q1", "동일"): 수익[n]["Q1"] for n in 결과} | {(f"NH퀄리티로우볼 {n} LS", "동일"): LS[n] for n in LS})
이상값 = {(f"NH퀄리티로우볼 {n} Q1", "동일"): int(이상[n][("동일", "Q1")]) for n in 결과} | {(f"NH퀄리티로우볼 {n} LS", "동일"): int(이상[n][("동일", "Q1")] + 이상[n][("동일", "Q5")]) for n in LS}
평가.저장(S, F, "03_NH퀄리티로우볼", [k for k in S if k[0].endswith(" LS")], 이상값=이상값)
pd.Series(이상값, name="이상값 보유 건수")